In [ ]:
import numpy as np
import scipy.stats as si

def black_scholes_call(S, K, T, r, sigma):
    
    d1 = (np.log(S/K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    call_price = S * si.norm.cdf(d1) - K * np.exp(-r * T) * si.norm.cdf(d2)
    return call_price

from scipy.stats import norm
from scipy.optimize import brentq

def implied_volatility(S, K, T, r, C):
    #Calculate the implied volatility of an option using the Black-Scholes formula.
    def f(sigma):
        d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
        d2 = d1 - sigma*np.sqrt(T)
        return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2) - C
    return brentq(f, 1e-6, 1)

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy import interpolate
from math import log, sqrt, exp

# Load option prices from CSV
df = pd.DataFrame({
    'Type': ['Call', 'Call', 'Call', 'Call', 'Call', 'Call', 'Call', 'Call', 'Call', 'Call'],
    'Strike': [2900, 2950, 3000, 3050, 3100, 3150, 3200, 3250, 3300, 3350],
    'Bid': [185.2, 156.2, 128.5, 102.6, 79.9, 60.1, 43.4, 29.1, 18.4, 10.4],
    'Ask': [187.7, 158.8, 130.8, 104.9, 81.8, 62.0, 45.3, 31.1, 19.8, 11.6],
    'DaysToExpiry': [28, 28, 28, 28, 28, 28, 28, 28, 28, 28]
})

# Set parameters for VIX calculation
S = 3000  # Underlying price
r = 0.01  # Risk-free interest rate
q = 0.0   # Dividend yield

# Filter options by days to expiry and type
df = df[(df['DaysToExpiry'] >= 21) & (df['DaysToExpiry'] <= 35) & (df['Type'] == 'Call')]

# Find the two strikes closest to the current price of the underlying
K1 = df['Strike'][df['Strike'] <= S].max()
K2 = df['Strike'][df['Strike'] >= S].min()

# Calculate the time to expiry in years
T = df['DaysToExpiry'].iloc[0] / 365

# Calculate the forward price
F = S * exp((r - q) * T)

# Calculate the option premiums and implied volatilities
df['Mid'] = (df['Bid'] + df['Ask']) / 2
df['IV'] = 0.0
for i, row in df.iterrows():
    K = row['Strike']
    mid = row['Mid']
    if K1 <= K <= K2:
        # Use Black-Scholes model to calculate implied volatility
        d1 = (log(F/K) + 0.5 * row['IV']**2 * T) / (row['IV'] * sqrt(T))
        d2 = d1 - row['IV'] * sqrt(T)
        C = F * exp(-q * T) * norm.cdf(d1) - K * exp(-r * T) * norm.cdf(d2)
        iv = implied_volatility(C, S, K, T, r)
        df.at[i, 'IV'] = iv
    else:
        # Use linear interpolation to calculate implied volatility
        iv1 = df['IV'][df['Strike'] == K1].iloc[0]
        iv2 = df['IV'][df['Strike'] == K2].iloc[0]
        iv = iv1 + (iv2 - iv1) * (K - K1) / (K2 - K1)
        df.at[i, 'IV'] = iv

# Calculate the weight for each option
df['Weight'] = df['Mid'] / (df['IV']**2 * T)

# Interpolate the implied volatilities to get the volatility for the at-the-money option
iv_atm = interpolate.interp1d(df['Strike'], df['IV'], kind='linear')(F)

# Calculate the VIX index
vix_squared = (df['Weight'] * df['IV']**2 * df['DaysToExpiry']).sum() * 2 / T - (F / S - 1)**2 / T
vix = sqrt(vix_squared) * 100

print('VIX:', vix)